# TL-Bot - char_classifier Training

**Before the very first run:**
- **Colab:** set runtime to GPU — *Runtime → Change runtime type → T4 GPU → Save*
- **Kaggle:** enable GPU (*Settings → Accelerator → GPU T4 x2*); add `RCLONE_TOKEN` secret (see below)
- **Lightning AI:** open a Studio with a T4 GPU before running this notebook
- **Local:** ensure `.venv` is active and a CUDA GPU is available (CPU works for smoke tests)

**Every session: run all cells top to bottom.**
- Cell 1 — set scripts and epoch count (platform is auto-detected).
- Cell 2 — mounts Drive (Colab), syncs checkpoints + dataset via rclone (Kaggle), confirms storage (Lightning/Local).
- Cell 3 — clones/pulls the repo and starts or resumes training.
- Cell 4 — Kaggle emergency fallback only; auto-sync runs during training.

Checkpoints are saved after every epoch and persist across sessions on all platforms.

---
**One-time: zip and upload the dataset**
```powershell
.venv\Scripts\python.exe Models\remote_train.py --zip-dataset
# Colab/Kaggle: upload char-dataset.zip to My Drive/Colab Notebooks/TL-Bot/
# Lightning:    upload char-dataset.zip to /teamspace/studios/this_studio/TL-Bot/
# Local:        dataset is already present — no zip needed
```

**One-time: Kaggle rclone setup**
```powershell
# 1. Configure rclone (if not already done)
rclone config   # → New remote → name: gdrive → type: Google Drive → follow OAuth flow

# 2. Copy the token JSON to your clipboard
(Get-Content "$env:APPDATA\rclone\rclone.conf" | Select-String "^token = ").ToString().Replace("token = ", "") | Set-Clipboard

# 3. Add a Kaggle Secret: Add-ons → Secrets → Add new secret
#    Name: RCLONE_TOKEN   Value: paste clipboard (the {"access_token":...} JSON)
```

---
## Cell 1 - Configure
Set the platform, scripts, and epoch count for this run. Edit here only.

In [4]:
from pathlib import Path as _Path

def _detect_platform():
    import os
    # Use env var — KAGGLE_KERNEL_RUN_TYPE is set by Kaggle's infrastructure.
    # /kaggle/input alone is unreliable: the kaggle Python package creates it on other platforms too.
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        pass
    if os.path.isdir("/teamspace/studios/this_studio"):
        return "lightning"
    return "local"

PLATFORM = _detect_platform()
print(f"Platform: {PLATFORM!r}")

# Scripts to train: "latin" | "kana" | "hangul" | "cjk" | "all"
# Single script -> checkpoints/<script>/   "all" -> checkpoints/
SCRIPTS = ["latin"]  #, "kana", "hangul", "cjk"]

# Epochs per run. Recommended: 60 per script, 80 for "all".
EPOCHS = 60

# --- Platform storage roots (only the one matching PLATFORM is used) ---

# Colab: Google Drive folder for checkpoints and dataset zip.
COLAB_ROOT = "/content/drive/MyDrive/Colab Notebooks/TL-Bot"

# Kaggle: rclone syncs checkpoints and dataset to/from the same Drive folder as Colab.
# See intro cell for one-time RCLONE_TOKEN setup.
KAGGLE_ROOT = "/kaggle/working/TL-Bot"

# Lightning AI: persistent studio storage path.
LIGHTNING_ROOT = "/teamspace/studios/this_studio/TL-Bot"

# Local: repo root (empty = cwd) and checkpoint output dir.
LOCAL_REPO = ""  # e.g. r"C:\Users\you\Documents\Discord-TL_Bot"
LOCAL_ROOT = str(_Path.home() / "tl-bot-checkpoints")

Platform: 'colab'


---
## Cell 2 - Setup
Mounts Drive (Colab), syncs checkpoints + dataset via rclone (Kaggle), or confirms local storage paths.

In [5]:
import os
if PLATFORM == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
elif PLATFORM == "kaggle":
    from kaggle_secrets import UserSecretsClient
    from pathlib import Path
    import subprocess
    import zipfile as _zipfile

    # Install rclone if not already present
    _which = subprocess.run(["which", "rclone"], capture_output=True)
    if _which.returncode != 0:
        print("Installing rclone ...")
        subprocess.run("curl -fsSL https://rclone.org/install.sh | sudo bash",
                       shell=True, check=True)
        print("rclone installed.")
    else:
        print(f"rclone already installed: {_which.stdout.decode().strip()}")

    # Configure gdrive via env vars — no config file needed.
    # RCLONE_TOKEN = the {"access_token":...} JSON from the "token = " line of rclone.conf.
    os.environ["RCLONE_CONFIG_GDRIVE_TYPE"]  = "drive"
    os.environ["RCLONE_CONFIG_GDRIVE_SCOPE"] = "drive"
    os.environ["RCLONE_CONFIG_GDRIVE_TOKEN"] = UserSecretsClient().get_secret("RCLONE_TOKEN").strip()
    r = subprocess.run(["rclone", "listremotes"], capture_output=True, text=True)
    if "gdrive:" not in r.stdout:
        raise RuntimeError("gdrive remote not found — check RCLONE_TOKEN secret.")
    print(f"rclone remotes: {r.stdout.strip()}")

    # Sync checkpoints from Drive (empty on first run is fine)
    ckpt_dst = Path(KAGGLE_ROOT) / "checkpoints"
    ckpt_dst.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(["rclone", "sync",
                             "gdrive:Colab Notebooks/TL-Bot/checkpoints/",
                             str(ckpt_dst), "--progress"])
    if result.returncode != 0:
        print("Warning: checkpoint sync returned non-zero — continuing (may be first run).")
    else:
        print("Checkpoint sync complete.")

    # Pull dataset from Drive (skip if already extracted this session)
    ds_dir = Path(KAGGLE_ROOT) / "char-dataset"
    if not ds_dir.exists():
        ds_zip = Path(KAGGLE_ROOT) / "char-dataset.zip"
        print("Pulling char-dataset.zip from Drive ...")
        subprocess.run(["rclone", "copy",
                        "gdrive:Colab Notebooks/TL-Bot/char-dataset.zip",
                        str(Path(KAGGLE_ROOT)), "--progress"], check=True)
        with _zipfile.ZipFile(ds_zip, "r") as zf:
            zf.extractall(Path(KAGGLE_ROOT))
        ds_zip.unlink()
        print(f"Dataset ready: {ds_dir}")
    else:
        print(f"Dataset already present: {ds_dir}")
elif PLATFORM == "lightning":
    os.makedirs(LIGHTNING_ROOT, exist_ok=True)
    print(f"Storage ready: {LIGHTNING_ROOT}")
elif PLATFORM == "local":
    from pathlib import Path
    Path(LOCAL_ROOT).mkdir(parents=True, exist_ok=True)
    print(f"Repo  : {LOCAL_REPO or os.getcwd()}")
    print(f"Ckpts : {LOCAL_ROOT}")
else:
    raise ValueError(f"Unknown PLATFORM: {PLATFORM!r}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
## Cell 3 - Train
Clones or pulls the repo (Colab/Lightning), then starts or resumes training.

In [ ]:
import os, subprocess, json
from pathlib import Path as _Path

REPO_URL = "https://github.com/alexjade96/Discord-TL_Bot.git"

if PLATFORM == "colab":
    REPO_DIR     = "/content/Discord-TL_Bot"
    STORAGE_ROOT = COLAB_ROOT
    storage_args = ["--storage-root", COLAB_ROOT]
elif PLATFORM == "kaggle":
    REPO_DIR     = "/kaggle/working/Discord-TL_Bot"
    STORAGE_ROOT = KAGGLE_ROOT
    storage_args = ["--storage-root", KAGGLE_ROOT, "--skip-dataset",
                    "--repo-dir", REPO_DIR,
                    "--sync-to", "gdrive:Colab Notebooks/TL-Bot/checkpoints/"]
elif PLATFORM == "lightning":
    REPO_DIR     = f"{LIGHTNING_ROOT}/Discord-TL_Bot"
    STORAGE_ROOT = LIGHTNING_ROOT
    storage_args = ["--storage-root", LIGHTNING_ROOT]
elif PLATFORM == "local":
    REPO_DIR     = LOCAL_REPO or os.getcwd()
    STORAGE_ROOT = LOCAL_ROOT
    storage_args = ["--storage-root", LOCAL_ROOT, "--skip-dataset",
                    "--repo-dir", REPO_DIR]
else:
    raise ValueError(f"Unknown PLATFORM: {PLATFORM!r}")

# Clone / pull for remote platforms; local repo is already present.
if PLATFORM in ("colab", "lightning", "kaggle"):
    os.makedirs(REPO_DIR, exist_ok=True)
    if os.path.isdir(f"{REPO_DIR}/.git"):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# Kaggle: symlink dataset into repo tree (Models/Datasets/ is gitignored, won't exist after clone).
if PLATFORM == "kaggle":
    _ds_src = str(_Path(KAGGLE_ROOT) / "char-dataset")
    _ds_dst = f"{REPO_DIR}/Models/Datasets/char-dataset"
    if not os.path.exists(_ds_dst):
        os.makedirs(f"{REPO_DIR}/Models/Datasets", exist_ok=True)
        os.symlink(_ds_src, _ds_dst)
        print(f"Dataset linked: {_ds_src} → {_ds_dst}")

# Print last training progress from progress.json before launching -- DO NOT REMOVE
def _print_progress(storage_root, scripts):
    _all = {"latin", "kana", "hangul", "cjk"}
    s = _all if "all" in scripts else set(scripts)
    if s >= _all:
        sub = "checkpoints"
    elif len(scripts) == 1:
        sub = f"checkpoints/{scripts[0]}"
    else:
        sub = "checkpoints/" + "_".join(sorted(s))
    p = _Path(storage_root) / sub / "progress.json"
    if not p.exists():
        print("[progress] No prior run found — starting fresh.")
        return
    try:
        d = json.loads(p.read_text())
        print("[progress]")
        for k, v in d.items():
            if isinstance(v, list):
                continue
            print(f"  {k}: {v:.6g}" if isinstance(v, float) else f"  {k}: {v}")
    except Exception as e:
        print(f"[progress] Could not read progress.json: {e}")

_print_progress(STORAGE_ROOT, SCRIPTS)

subprocess.run(
    [
        "python", "-u", f"{REPO_DIR}/Models/remote_train.py",
        "--skip-clone", "--resume",
        "--scripts", *SCRIPTS,
        "--epochs", str(EPOCHS),
        *storage_args,
    ],
    check=True,
)

[progress]
  backbone: dinov2_vits14
  total_epochs: 60
  freeze_epochs: 5
  completed: 29
  epochs_remaining: 31
  phase: 2
  phase_label: backbone fine-tune
  best_val_acc: 0.535561
  best_epoch: 24
  epochs_since_best: 5
  last_val_acc: 0.497515
  last_val_acc_delta: -0.017995
  last_f1: 0.539534
  last_precision: 0.632959
  last_recall: 0.496975
  last_lr: 0.00056983
  last_grad_norm: 4.9757
  last_epoch_secs: 1096.8
  eta_secs: 34001
  saved_at: 2026-07-22T22:48:02


---
## Cell 4 - Sync (Kaggle emergency fallback)
On Kaggle, checkpoints are automatically synced to Drive every 10 minutes during training — **you do not need to run this cell under normal conditions.**

Run it manually only if the session crashed before the next auto-sync and you want to push whatever checkpoints exist right now.

In [ ]:
if PLATFORM == "kaggle":
    import subprocess
    from pathlib import Path
    ckpt_src = Path(KAGGLE_ROOT) / "checkpoints"
    print(f"Syncing {ckpt_src} → Drive ...")
    subprocess.run([
        "rclone", "sync", str(ckpt_src),
        "gdrive:Colab Notebooks/TL-Bot/checkpoints/",
        "--progress",
    ], check=True)
    print("Done.")